In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(5, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [3]:
from processor import PolarsLoader, ExprProcessor, PandasConverter

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
# import shutil
# shutil.rmtree('exp/exp1')
import os

In [7]:
from modeler import Experimenter
from modeler.collector import MetricCollector, ModelAttrCollector
from modeler import Connector
from sklearn.metrics import roc_auc_score

if os.path.exists('exp/exp1'):
    e = Experimenter.load('exp/exp1', df_train)
else:
    e = Experimenter.create(
        df_train, 'exp/exp1', sp = StratifiedShuffleSplit(n_splits=1, random_state = 123), 
        sp_v = StratifiedShuffleSplit(n_splits=1, train_size=0.9, random_state = 123), splitter_params = {'y': y}
    )

Loaded: 29 node(s), 6 group(s), 1 fold(s)


In [8]:
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

# Configuration

In [9]:
e.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, y)]}), '.*'+ y +'_1', roc_auc_score, include_train = True
    )
)
e.add_collector(
    ModelAttrCollector(
        'lgb_feature_importance', Connector(processor = lgb.LGBMClassifier), 'feature_importances'
    )
)

e.set_grp('clf', role = 'head', method = 'predict_proba', edges = {'y': [(None, y)]})
e.set_grp('lgb', parent = 'clf', processor = lgb.LGBMClassifier, params={'verbose': -1, 'early_stopping': lgb.early_stopping(100), 'eval_metric': 'AUC'})
e.set_grp('xgb', parent = 'clf', processor = xgb.XGBClassifier)
e.set_grp('cb', parent = 'clf', processor = cb.CatBoostClassifier)
e.set_grp('lr', parent = 'clf', processor = LogisticRegression)
e.set_grp('pre', role = 'stage', method = 'transform')

{'result': 'skip',
 'grp': <modeler._pipeline.PipelineGroup at 0x7f8269f30f20>,
 'affected_nodes': []}

## Stage Nodes

In [10]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
e.set_node(
    'ord', grp = 'pre', processor=OrdinalEncoder, 
    edges ={'X': [(None, 'grade_subgrade')]}, params={'categories': [np.sort(df_train['grade_subgrade'].unique())]}
)

e.set_node(
    'ohe', grp = 'pre', processor=OneHotEncoder, 
    edges ={'X': [(None, X_cat)]}, params={'sparse_output': False}
)

e.set_node(
    'std', grp = 'pre', processor=StandardScaler, edges ={'X': [(None, X_num)]}
)
e.build()

Building 0 node(s)
Build 1/1 (100%) Node 0
Build complete: 0 node(s)


## LightGBM

In [11]:
e.set_node('lgb1', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.1})
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [12]:
e.set_node(
    'lgb2', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05}
)
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [13]:
e.add_collector(
    ModelAttrCollector(
        'lgb_evals_results', 
        Connector(processor=lgb.LGBMClassifier),
        'evals_result'
    )
)

In [14]:
e.set_node(
    'lgb3', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.2}
)
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [15]:
e.set_node(
    'lgb4', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075}
)
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [16]:
e.set_node(
    'lgb5', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075, 'num_leaves': 15}
)

{'result': 'skip',
 'affected_nodes': [],
 'old_obj': <modeler._pipeline.PipelineNode at 0x7f827b3d24e0>,
 'obj': <modeler._pipeline.PipelineNode at 0x7f827b3d24e0>}

In [17]:
e.set_node(
    'lgb6', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.1, 'num_leaves': 15}
)
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [18]:
e.set_node(
    'lgb7', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05, 'num_leaves': 15}
)
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [19]:
e.set_node(
    'lgb8', grp = 'lgb', edges = {'X': [(None, X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05, 'num_leaves': 15}
)
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [20]:
e.set_node(
    'lgb9', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, 
    params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075, 'num_leaves': 63}
)
e.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb9 0/1 (0%) > 1/10000 (0%) training-auc: 0.9103, training-binary_logloss: 0.4636, valid_1-auc: 0.9117, valid_1-binary_logloss: 0.4637Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[403]	training's auc: 0.938185	training's binary_logloss: 0.223982	valid_1's auc: 0.922788	valid_1's binary_logloss: 0.243669
Exp 1/1 (100%) lgb9 1/1 (100%)
Experimentation complete: 1 node(s)


In [21]:
d = e.collectors['lgb_evals_results'].get_attrs('lgb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('valid_1', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
    }, name = ('evals_result', 'best_iteration'))
e.pipeline.compare_nodes(
    e.pipeline.get_node_names('lgb*')
)['LGBMClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending = False)

params             \
     learning_rate num_leaves   
lgb5         0.075       15.0   
lgb6         0.100       15.0   
lgb7         0.050       15.0   
lgb2         0.050    default   
lgb1         0.100    default   
lgb4         0.075    default   
lgb9         0.075       63.0   
lgb3         0.200    default   
lgb8         0.050       15.0   

                                                                                         X  \
     DataSource [education_level, employment_status, gender, loan_purpose, marital_status]   
lgb5  [annual_income, credit_score, debt_to_income_r...                                      
lgb6  [annual_income, credit_score, debt_to_income_r...                                      
lgb7  [annual_income, credit_score, debt_to_income_r...                                      
lgb2  [annual_income, credit_score, debt_to_income_r...                                      
lgb1  [annual_income, credit_score, debt_to_income_r...                                      
lgb4  [annual_income, credit_score, debt_to_income_r...                                      
lgb9  [annual_income, credit_score, debt_to_income_r...                                      
lgb3  [annual_income, credit_score, debt_to_income_r...                                      
lgb8                                                 []                                      

           AUC                       evals_result  
         valid train_sub valid_sub best_iteration  
lgb5  0.924416  0.933446  0.923891         1288.0  
lgb6  0.924324  0.933689  0.923844          994.0  
lgb7  0.924239  0.935235  0.924013         2305.0  
lgb2  0.923872  0.939157  0.923553         1400.0  
lgb1  0.923860  0.936376  0.923525          562.0  
lgb4  0.923789  0.933892  0.923030          628.0  
lgb9  0.923561  0.938185  0.922788          402.0  
lgb3  0.922969  0.931672  0.922612          204.0  
lgb8  0.787390  0.789670  0.785937          145.0

## XGB

In [22]:
e.add_collector(
    ModelAttrCollector('xgb_feature_importance', Connector(processor=xgb.XGBClassifier), 'feature_importances', params={'importance_type': 'gain'})
)
e.add_collector(
    ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result')
)

# XGB with preprocessed stage features (ohe + std + ord)
e.set_node('xgb1', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.set_node('xgb2', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.set_node('xgb3', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc', 'max_depth': 4})
e.set_node('xgb4', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.075, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
# xgb5: ohe 제외 (categorical feature 없이)
e.set_node('xgb5', grp='xgb', edges={'X': [('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [23]:
d = e.collectors['xgb_evals_results'].get_attrs('xgb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_1', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))

e.pipeline.compare_nodes(
    e.pipeline.get_node_names('xgb*')
)['XGBClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending=False)

params                 AUC                       evals_result
     learning_rate max_depth     valid train_sub valid_sub best_iteration
xgb3         0.100       4.0  0.923342  0.930310  0.922908         1111.0
xgb2         0.050   default  0.922984  0.933137  0.922393          810.0
xgb4         0.075   default  0.922855  0.930193  0.922106          392.0
xgb1         0.100   default  0.922743  0.933216  0.922059          393.0
xgb5         0.100   default  0.806164  0.826120  0.804998          387.0

## Catboost

In [24]:
e.add_collector(
    ModelAttrCollector('cb_feature_importance', Connector(processor=cb.CatBoostClassifier), 'feature_importances_pvc')
)
e.add_collector(
    ModelAttrCollector('cb_evals_results', Connector(processor=cb.CatBoostClassifier), 'evals_result')
)

# CB with raw features (native categorical handling)
e.set_node('cb1', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb2', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb3', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.1, 'depth': 4, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb4', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.075, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
# cb5: grade_subgrade 추가
e.set_node('cb5', grp='cb', edges={'X': [(None, X_num + X_cat + ['grade_subgrade'])]},
    params={'cat_features': X_cat + ['grade_subgrade'], 'iterations': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [25]:
d = e.collectors['cb_evals_results'].get_attrs('cb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_1', 'AUC')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))

e.pipeline.compare_nodes(
    e.pipeline.get_node_names('cb*')
)['CatBoostClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending=False)

params                         \
                                          cat_features    depth learning_rate   
cb3  [gender, marital_status, education_level, empl...      4.0         0.100   
cb4  [gender, marital_status, education_level, empl...  default         0.075   
cb1  [gender, marital_status, education_level, empl...  default         0.100   
cb2  [gender, marital_status, education_level, empl...  default         0.050   
cb5  [gender, marital_status, education_level, empl...  default         0.100   

                                                                                                                                                                       X  \
    DataSource [annual_income, credit_score, debt_to_income_ratio, education_level, employment_status, gender, interest_rate, loan_amount, loan_purpose, marital_status]   
cb3                                                 []                                                                                                                     
cb4                                                 []                                                                                                                     
cb1                                                 []                                                                                                                     
cb2                                                 []                                                                                                                     
cb5                                   [grade_subgrade]                                                                                                                     

          AUC                       evals_result  
        valid train_sub valid_sub best_iteration  
cb3  0.925199  0.927305  0.924605         2650.0  
cb4  0.925021  0.928971  0.924266         1921.0  
cb1  0.925007  0.930206  0.924551         1687.0  
cb2  0.924933  0.930010  0.924438         3335.0  
cb5  0.924929  0.929659  0.924333         1708.0

## Logistic Regression

In [26]:
from modeler import col

In [27]:
for i, C in enumerate([1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3]):
    e.set_node(f'lr{i}', grp='lr', edges={'X': [('std', None), ('ohe', col.ohe_drop_first)]}, params={'C': C})
e.exp()

Experimenting 0 node(s)
Exp 1/1 (100%) Node 0
Experimentation complete: 0 node(s)


In [28]:
e.pipeline.compare_nodes(
    e.pipeline.get_node_names('lr*')
)['LogisticRegression'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
)

params       AUC                    
            C     valid train_sub valid_sub
lr0     0.001  0.910424  0.908343  0.909686
lr1     0.010  0.911870  0.909773  0.910948
lr2     0.100  0.912058  0.909939  0.911177
lr3     1.000  0.912075  0.909947  0.911209
lr4    10.000  0.912080  0.909947  0.911212
lr5   100.000  0.912081  0.909947  0.911213
lr6  1000.000  0.912081  0.909947  0.911213

In [29]:
from IPython.display import Markdown
Markdown(
    e.desc_node('lr1', show_params=True)
)

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr1["clf/lr/lr1"]
        lr1_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr><tr><td align='left'><b>C</b></td><td align='left'>0.01</td></tr></table>"]
    end
    style node_lr1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["pre/ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["pre/std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr></table>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    DataSource -->|y| node_lr1
    DataSource --> node_ohe
    DataSource --> node_std
    node_ohe --> node_lr1
    node_std --> node_lr1
```

**Path from DataSource to 'clf/lr/lr1' (3 path(s) found)**

### Edges

| Key | Node | Var |
|-----|------|-----|
| X | pre/std | * |
| X | pre/ohe | `<function ohe_drop_first at 0x7f827b3da840>` |
| y | Data Source | `loan_paid_back` |

In [30]:
Markdown(
    e.desc_pipeline(max_depth = 2)
)

```mermaid
graph TD

    DataSource([DataSource])
    style DataSource fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_pre["pre"]
        node_ord["ord"]
        style node_ord fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_pre fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lgb["lgb"]
            grp_lgb_count["9 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_xgb["xgb"]
            grp_xgb_count["5 node(s)"]
            style grp_xgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_xgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["5 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lr["lr"]
            grp_lr_count["7 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    DataSource --> grp_clf
    DataSource --> grp_pre
    grp_pre --> grp_clf
```

In [31]:
e.close_exp()

Finalize 'std'
Finalize 'ohe'
Finalize 'ord'
Finalize 'lgb1'
Finalize 'lgb2'
Finalize 'lgb3'
Finalize 'lgb4'
Finalize 'lgb5'
Finalize 'lgb6'
Finalize 'lgb7'
Finalize 'lgb8'
Finalize 'cb4'
Finalize 'cb3'
Finalize 'cb2'
Finalize 'cb5'
Finalize 'cb1'
Finalize 'xgb4'
Finalize 'xgb5'
Finalize 'xgb2'
Finalize 'xgb3'
Finalize 'xgb1'
Finalize 'lr1'
Finalize 'lr4'
Finalize 'lr2'
Finalize 'lr6'
Finalize 'lr0'
Finalize 'lr3'
Finalize 'lr5'
Finalize 'lgb9'
